# README
- **Author**: RUEI-HSIANG, CHANG (B12303030)，Codex 協助
- **Created At**: 2026-10-19
- **Last Modified At**: 2026-10-19

## What does this file do?
擷取已保存競選／政黨人物頁的學歷、一般經歷與人物入口；原文分欄、不推定學位完成。以姓名＋縣市＋公職唯一對應研究名冊；這不等於完成身分獨立複核。四筆明列過去參選／任職者另記首次任何公職=0；其中于仙玲另有前兩屆議員參選證據。

## What does this file take?
固定來源與 SHA-256：程式 PINS。原始來源說明 data/raw/social_sources_20261019_documentation.ipynb；既有主表、欄位表與692筆 MOI 人物頁名單保留不變。

## What does this file output?
- data/processed/candidate_background_evidence_round3_20261019.csv：逐人逐欄原文及來源，collection evidence。
- data/processed/council_campaign_profiles_20261019.csv：本輪議員競選人物頁，collection index。
- data/processed/candidate_master_reviewed_round3_20261019.csv：1,583人更新狀態快照。
- data/processed/candidate_fields_reviewed_round3_20261019.csv：31,660欄位清單更新快照。
全部由保存檔讀回、檢查鍵、schema、列數及 hash。文件 data/processed/background_round3_20261019_documentation.ipynb。
一般經歷不等於行政經歷；不從照片、姓名、學校或代名詞推測性別，不把研究所／在學當作已取得學位，不把歷任民代自動當作2026仍現任。


In [1]:

from pathlib import Path
import hashlib,json,re,unicodedata
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'templates').is_dir())
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()
PINS={'data/raw/round3_20261019/capture_manifest.json': 'a701143ad22134673e670515332b9691ba9d3f911c9903671e290079509de24f', 'data/processed/candidate_master_reviewed_20261019.csv': '8beb9e6d8c766f9b1484f5c2bc71a1bbf3702640be2135a3abb907d1e6e96177', 'data/processed/candidate_fields_reviewed_20261019.csv': 'ff10c75856103a246126dd38ce4b274c647265dc7f4c1241cfcd6f5daae3e41c', 'data/processed/candidate_profile_leads_20261019.csv': '52b723bdeea132df0f5ef9f9ac97594993784ba84edb796c82f0959d63236a3f'}
for p,h in PINS.items():assert sha(ROOT/p)==h,p
manifest=json.loads((ROOT/'data/raw/round3_20261019/capture_manifest.json').read_text())
lookup={r['url']:r for r in manifest}
for r in manifest:
    if r['path']:assert sha(ROOT/r['path'])==r['sha256']
master=pd.read_csv(ROOT/'data/processed/candidate_master_reviewed_20261019.csv',dtype=str).fillna('')
fields=pd.read_csv(ROOT/'data/processed/candidate_fields_reviewed_20261019.csv',dtype=str).fillna('')
old=pd.read_csv(ROOT/'data/processed/candidate_profile_leads_20261019.csv',dtype=str).fillna('')
def norm(s):return re.sub(r'\s+','',unicodedata.normalize('NFKC',s)).replace('台','臺').replace('温','溫')
def match(name,city,mayor):
    d=master[master.candidate_name.map(norm).eq(norm(name)) & master.county_city.map(norm).eq(norm(city)) & master.office_type.str.contains('長' if mayor else '議員')]
    return d.iloc[0].to_dict() if len(d)==1 else None
def soup(r):return BeautifulSoup((ROOT/r['path']).read_bytes(),'html.parser')
rows=[];profiles=[]
def add(person,field,value,rec,locator):
    if not value:return
    value=str(value).replace('\r\n','\\n').replace('\n','\\n')
    rows.append(dict(candidate_key=person['candidate_key'],candidate_name=person['candidate_name'],county_city=person['county_city'],office_type=person['office_type'],field=field,value_raw=value,source_url=rec['url'],source_file=rec['path'],source_sha256=rec['sha256'],source_locator=locator,evidence_tier='campaign_or_party_declared_not_independent_certification',identity_status='unique_name_county_office_match_pending_additional_identity_review',observation_date='2026-10-19'))
index=lookup['https://www.taiwangogo.tw/candidates/']
for card in soup(index).select('a.cl-cd[href]'):
    name=card.select_one('.nm').get_text(strip=True);city=card['data-city'];person=match(name,city,False)
    rec=lookup.get(urljoin(index['url'],card['href']))
    if not person or not rec or rec['http_status']!='200':continue
    s=soup(rec);bio=s.select_one('.cv-biocard')
    if bio is None:continue
    vals={}
    for h in bio.select('h4'):
        label=re.sub(r'\s+','',h.get_text());ul=h.find_next_sibling('ul')
        if label not in ['學歷','經歷'] or ul is None:continue
        vals[label]=' | '.join(x.get_text(' ',strip=True) for x in ul.find_all('li',recursive=False))
    add(person,'profile_url',rec['url'],rec,'candidate card name/city/office + candidate detail page')
    for label,field in [('學歷','education'),('經歷','experience')]:add(person,field,vals.get(label,''),rec,f'.cv-biocard h4={label} + sibling ul li')
    profiles.append(dict(candidate_key=person['candidate_key'],candidate_name=person['candidate_name'],county_city=person['county_city'],office_type=person['office_type'],profile_url=rec['url'],education_raw=vals.get('學歷',''),experience_raw=vals.get('經歷',''),birth_date='',sex='',birth_date_status='not_listed_in_background_block',sex_status='not_listed_in_background_block',identity_status='unique_name_county_office_match_pending_additional_identity_review',previous_moi_profile_present=person['candidate_key'] in set(old.candidate_key),source_file=rec['path'],source_sha256=rec['sha256']))
js=next(r for r in manifest if 'script_mayor.js' in r['url']);t=(ROOT/js['path']).read_text();data=json.JSONDecoder().raw_decode(t[t.index('['):])[0]
for item in data:
    person=match(item['name'],item['city'],True)
    if not person:continue
    add(person,'profile_url','https://teamtaiwan.dpp.org.tw/',js,f"mayor_data name={item['name']}")
    add(person,'education',' | '.join(item.get('education',[])),js,f"mayor_data name={item['name']} field=education")
    add(person,'experience',' | '.join(item.get('experience',[])),js,f"mayor_data name={item['name']} field=experience")
evidence=pd.DataFrame(rows);profiles=pd.DataFrame(profiles)
assert not evidence.duplicated(['candidate_key','field','source_url']).any()
assert profiles.candidate_key.is_unique
# Explicit, inspected prior-candidacy declarations only; do not infer first=1 from silence.
prior={'謝閔弘':'2022 臺北市北投區裕民里里長候選人','陳峙穎':'2018-2022年','林裙靜':'2022 竹東鎮第四選區鎮民代表參選人','于仙玲':'2018、2022年小民參政歐巴桑聯盟黨前鎮小港區市議員候選人'}
additional=[]
for name,quote in prior.items():
    d=evidence[evidence.candidate_name.eq(name)&evidence.field.eq('experience')];assert len(d)==1 and quote in d.iloc[0].value_raw
    r=d.iloc[0].to_dict();r.update(field='first_time_any_office',value_raw='0',source_locator='背景經歷明列曾參選／擔任民選公職：'+quote);additional.append(r)
    if name=='于仙玲':r=r.copy();r.update(field='first_time_this_office',source_locator='背景經歷明列2018、2022年市議員候選人');additional.append(r)
evidence=pd.concat([evidence,pd.DataFrame(additional)],ignore_index=True)
# Only update existing fields corresponding exactly to the evidence meaning.
# General experience is NOT automatically administrative experience.
statusmap={'profile_url':'profile','education':'education','first_time_any_office':'first_time_any_office','first_time_this_office':'first_time_this_office'}
for r in evidence.to_dict('records'):
    if r['field'] not in statusmap:continue
    f=statusmap[r['field']];mask=fields.candidate_key.eq(r['candidate_key'])&fields.field.eq(f)
    assert mask.sum()==1
    fields.loc[mask,['existing_value','evidence','status','note','next_source','updated_date']]=[r['value_raw'],r['source_url'],'新增競選／政黨原文，待獨立複核','來源申報；未認證學位、身分或2026仍在任。原版表保留。','官方人物頁／公報及身分交叉核對','2026-10-19']
    mm=master.candidate_key.eq(r['candidate_key']);assert mm.sum()==1
    master.loc[mm,f+'_status']='新增競選／政黨原文，待獨立複核'
assert len(master)==1583 and len(fields)==31660
assert not fields.duplicated(['candidate_key','field']).any()
outs={'data/processed/candidate_background_evidence_round3_20261019.csv':evidence,'data/processed/council_campaign_profiles_20261019.csv':profiles,'data/processed/candidate_master_reviewed_round3_20261019.csv':master,'data/processed/candidate_fields_reviewed_round3_20261019.csv':fields}
verification={}
for path,frame in outs.items():
    p=ROOT/path;assert p.parent==ROOT/'data/processed' and path not in PINS
    frame.to_csv(p,index=False,encoding='utf-8-sig');d=pd.read_csv(p,dtype=str).fillna('')
    assert len(d)==len(frame) and list(d.columns)==list(frame.columns)
    assert len(p.read_text(encoding='utf-8-sig').splitlines())==len(d)+1
    verification[path]={'rows':len(d),'sha256':sha(p)}
new=profiles[~profiles.previous_moi_profile_present]
print(json.dumps({'council_profiles':len(profiles),'new_council_profiles':len(new),'original_810_still_without_this_profile_source':810-len(new),'mayor_profiles':sum(evidence[evidence.field.eq('profile_url')].office_type.str.contains('長')),'evidence_rows':len(evidence),'outputs':verification},ensure_ascii=False,indent=2))
(ROOT/'data/code/background_round3_20261019_verification.json').write_text(json.dumps(verification,ensure_ascii=False,indent=2))
print('謝閔弘學歷：',profiles[profiles.candidate_name.eq('謝閔弘')].iloc[0].education_raw)
assert '在學' in profiles[profiles.candidate_name.eq('謝閔弘')].iloc[0].education_raw
assert profiles.birth_date.eq('').all() and profiles.sex.eq('').all()


{
  "council_profiles": 28,
  "new_council_profiles": 25,
  "original_810_still_without_this_profile_source": 785,
  "mayor_profiles": 19,
  "evidence_rows": 146,
  "outputs": {
    "data/processed/candidate_background_evidence_round3_20261019.csv": {
      "rows": 146,
      "sha256": "5195c8b1902f8228f8001f3986e7d2d433b0662e2d5a069bdd015a491d4bf663"
    },
    "data/processed/council_campaign_profiles_20261019.csv": {
      "rows": 28,
      "sha256": "28fed78971de8248fab91ce32c889851ae2df17de303c2ea25b3e0bb9778ae00"
    },
    "data/processed/candidate_master_reviewed_round3_20261019.csv": {
      "rows": 1583,
      "sha256": "b5656f830d321c7806d3742175f7a0761699f59f3ecd442cb9a5b3310b1c8840"
    },
    "data/processed/candidate_fields_reviewed_round3_20261019.csv": {
      "rows": 31660,
      "sha256": "72c245eedc0396077cf2fa5c0e821f68c41c9d566d10242b3bb2ba5b2e615144"
    }
  }
}
謝閔弘學歷： 石牌國小 | 明德國中 | 建國中學 | 國立臺灣師範大學東亞學系在學
